# ATLAS — FMNIST @107M · Stack v2 (3 configs)

**Self-contained** — não precisa clonar repo nem token GitHub.

**Como usar:**
1. Runtime → **GPU (A100 ou H100)**
2. **Run all** (~**35–50 min** total)

**3 experimentos × 3 seeds = 9 treinos:**

| ID | Stack | O que testa |
|----|-------|-----------|
| `exp-100m-wc` | **warmup+cosine** | Receita campeã CPU (89,93%) — confirmar @107M GPU |
| `exp-100m-ms` | multi_scale + LS + wc | Campeão CIFAR (+3,48 pp) |
| `exp-100m-tri` | tri_scale + LS + wc | Recorde CPU pequeno (92,98%) |

**Escala:** `width_mult=16`, `hidden_dim=2048`, ~107M params, 1000 steps, bf16.

**Referência CPU (não reroda):** warmup+cosine **89,93%** | baseline plain **86,12%**


In [ ]:
# @title 1. GPU
import torch
assert torch.cuda.is_available(), "Ative GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("PyTorch:", torch.__version__)

In [ ]:
# @title 2. ATLAS trainer (self-contained — sem GitHub)
import base64, gzip, pathlib, sys, importlib

TRAINER_B64 = """H4sIAHnhS2oC/+09227kyHXv+gq6jQDkiKLULWkugnvhWc/O2snM7MQaO0CUAUE1q9WMeDPJbrXGWSD/kPxAkgfDAfwSIC95zPxJviTnnLoX2a2WRmv7wcZ6mmSdOlV16tS5VpV+/KPDZdscXmblIStXXn3bLaryeG80Gn1okqxkjZcyr2Fz1nz+z3KWJV7x+Q9lVlTeyw9vXp57//fP/+rVSTdbsNZL6jybJWnVeiuAm+fJVRt65bKcJYCAtbMm6xKvq7okjwD93t68qQovjufLbtmwOPayoq6azkvKEmC6rCrbvT35rbmqk6Zl8v0f26qUz0XSLeRzk5RpVci3LisYbyVNumSWJ20L3ZQo2zSbdaEuCr15xvJUVWBYXULL95CQfqpKgRjGvsizSwn2HvtCBd1tnZVX8vvL8laNpaua2cJ6icoSuuOVpfs1mgP1kBJJjgCvBWoqXXZZ3kbYe9nIK3h+UyUpawy4VdZCfWMYScs6GGsHpGrnVVMAkfd+qoiwR/96NPc/q8p5dnW258H/WsbSMy8rO2/qnUz4p47Vrfz2/OiIPl4iL8Rt9onJkvHkOZXkzRnwRJXQN3ZwTB9vWHa16OKUzZJbq/iEihdZmrIyTrPCxZY2VV0tO13nKBrzZpJLlsdtUVXdAibABOA9vEmaYlnHVu95yVWTpPEsz+p+JV7ESqBaTmO7rKocyl8nOTAlQvxYLQNaGPStqoFZABxG3nYNgI+SNClGCIwP3j/Rzw38tldpXFQFNLAsOHEBVbrMjaol8BxVxQeoMqvajB7a3zQd1UmAVVa0cFSlhuVLqoQPAHvFf9qM/0CXsySPr4C540YWXZUsjREpvK2SJkvKGSOQlGCoqRIYZ2PPiAcQAp7z5JY14hl4LZ6DJCAMRbam6dmAI69m0LHLnJWp/RZfn8KHYpl3WdzCV6aAusb9wknEhye/4RxW9a39sa7aLgby6ZoJYEoFACcurCXV21lZUmfhF4mUIakWt5dNlho14LPJuU9P9FdWdwv5faI+L1iSKpbU0EApk4O6jreNDzjq6hpWiKKBfothZSOKrAQcRBdjgQm2ZkXirj3J8dAqrJIkrxdJv3C27GDxWcv8SKwChk0l3s/HQiCkbJXNmO4+1OTdhwckdI1MN1umCadyUdtrC0Ev5+OnBA+z0nn+t+9/FfCVnKXdIkZWsEaGVYg/SB3BVJZJ1npp5Z0XSZ7/7N07z78EIZgDa7SgPLKqYS1HOKuKOstZvwdc4opiPiwQmDC7el6iQ/w04qtjWcQ3VXPNGkfC5NVVzFasuVWTL6jEVsDfvZJNgvmXrMVB24LZkclC1uV5jAorbgWJNFRcsyZu2cwsmGegaeIO24jzqm03lSUzq9olg+WDI3C+8yoDBeIT4M+ztruggo8waFLAfsrmCYwvnsOKrJrbKcLICSKN5KHmvgDCh6hYN1dEsACoCN+BTsCvQCtfESzwDr7y3oG44YTkpkOkYHiLfOaLpFxCl40yjvSKkdgQUtcvk4JzOqEGC4BdgA5/W6Eg/yhaYWDslN5vuWQ+A50f/ZK9+VXoja7Uh2+/oQ8opPmH8+zNr76/QOwfRcNJXee3ZtPrM9HVD6xsKyCM0xejjHckmxOINwXe7emBEYcRcOsIxJgfIOiJLiC2Bfvw6hbov46Sy9YPooIlpQ/QU38SesdB6F0zVuP7h2bJAqsu2JRQkWPQ9cZOlQjYvqj9IiunYBWc2iiwv4DDF904RJyyBoiq0JtER26jRP7XEY7SXwfeE0Kyt6m8TyqtHg0iiWrrCIt9xCoJgoB+H82AXt2V5lCVCA6/GykdesvyMgMhl05Jhg2SjfME9LiostRHtE+8k0chmPg4sDoCH5cOF2VvUKl/jarKV6sk4GMF9+Ddd7/+5s0ZqKEWcCXcAvBIcd5kIJRBFvNxcH3TgApvSNQL34Krnjm4F6T/Yli2+Tz0ZgvwLljORSTSrIE3KXKPQ0IJhliqFRZy5sgRFSRDlyA/YYJVA5pudZJCTY7bOzwUGp4L63wepTdQCiMG83o1SX3ZJd052a8QMaVgJU3hF/rWVMu6nWoonGF3foF11BiI03LwyhLw28qRzUY3igM6kgr+xQUtmTH+A08fQ++Cng9OxGf8YoN8tHlFjC7iRj13TabeTbTK2I0PCxv+O0ZeBV6pWdIZI+eFY40OvjK7v0AwJDQsiyaZ8enUbQU2hQ0KaGoYPIHfBEM4cnODsBSUdZFL9jgbWjL28lrfIRqDO9qB5blbM/cSC+6KH247Bd3nNF6QCLqHtF99ocwaHG2BA3DEmDukB86GYhZwkcFjTO/NLlxgTeWikKLRkL9Uwnlx3esvwTwRWPY9H+3aA/pK2kUJ0bdo6J6jdX+XJJ0v28//UYFIAReWgYWuZenx+hiaOF2faqkKM198/n2aJSBfaxQf9xCr9xOWgj7Hd8rEYy0OxzsKQ4H79E7cpxr3ZEfcw/LouCeQhuFOFdyjMdpRdAqcIfuBqnrf/nY6wIMPFVL35NIPzePx6L73bP3sL5y6O6cK3M/uxP1M4z7eETcwLeiCrPTUAgjVWNTTs2CrKi8efy349jIwFoB+ewZvAfgMxyKm8cdfFT+j8NS38P2udSE1hujRDFzzrFAMTzEt5q09NIvLFY3rAebwg1bDD7EY7jD1fkjVnFboCPhrOX3gwC2LHUwcinCihYMPm+El2rvA9jEE/nQ7W2JXD6nhBzPgNzwWujMHpqwBZy6tUOjyOKohdr008YrkCiZuCcV/4b8H8R8QUEdTtEMJ37DkEP/dzpF2vOSp4VGRi3vg18AB8A9x2PMgyqsrP9iVywf5EBE/sSMt92TD91XbvZx1d7Egj3G/fP/5X849jJlR7pGnEWAZEB9+/h3xZoIxh8+/A1PiL9zX4z4RkWEYo7DDNkO8qaDv4AIF97jKUqHVSpNnZu5ilrTKWs4xrccKr/38+yZjxDENw7A0ptrA0Gyqq4aBMl39sJxCHRkD0YyYlxvtmZrhp6mIOw2gmTwEzRdzjZgtoxu+MTI0OURkukiuWcxXpY6HD1FQzV0/Hm2k+/pB1geRcAP22A6r3LsFrL69gevT++E/3WkEMqb35TTS4cF+Mxj3MfKkX9IMhZD6LfTSuP023ACHbKiPzMn/9lHZXuhmRL20cR+Va7pvRtbPN/ex9eywzejsTHUflaVLtwzRzG8PDM8UsjYWAQGr9xcpjC3rbo2Q/i/fnr9uGNsilwEChAjum2k+/3vBYNytl8zBmUHRLHLHG2zHh4ouI5vyo142RWZuNAWKliywurrxJ3cltER6Z2OSSmLH1BRttBHhEMzS36XBKJUv7RskGXBHSUYN0YpbQzpAMn7ltdUluIFUr9UBkqu8ugRScjq+5DlxjKy0s+qSNV1ibbVILDtKoIKnZYGZZ75hiAnauqkZCb7KPmXlAubVR290bHQSHZte0gaIKaZYOheJjOiYA4q8v2dNhZu9muwTa72//ZtfEx2+898FqOFX7BP2AV7/97+CSNJyo0aXGzN6OaAvtAPHKXIL4e+ncYyUkDIGCfBPYQf+GGH9r0PvXei90s0uiPtpdxjIGoZ5m0mA0Aj6CqA324qLwK23c2DFTfuqFsdW5+4bdlTr7O35thwjqpgDUh1i1fHFgOzFp6z1jj3mnQqGBMZkv1mK7ZCK76Nd2O1LI44Wew06Fxv5qR9itJANBhbvyZx/guj3ILv244FOcHwxEBxfBI/Ivg/l2F9nH74GoOset24VYXrjWOjp3WIP4DWMJo05g7zBjXvvRJjKAaM2Ym5M8AZNZUsfyMqgvWpn/RyxgCAXANce9v9l16E5UZWcJ2lIodhOOs+atnNiESy3mjI2vm1r0dC81rD6+eYN1UCQ9OmBZJvsQLa85kDnID5wsEmO8/wGLEmeF6U5xQQm2BliH5DPHzUMFhNc8EirR0+8GQeQmWBzqodndBF6sURD0D584f+VjKUxX+o9CdKn+MLBomHXuMhh2Sz6htW+Iq2vp0K4o0IJZOUtLKsh+Q+f1XZ2zDl5r5N2ASz49t0vzj+c8Q293ska6H25TK9YRxsEk+7zH1agxdHAkjsKtwcP5ldn5obq+61LnPIp4ojEflKbp3gfzTDUWAh06DUIgQy8rxPSppPnB189g88vhL7SMjoWhhsYxy8c7FXLcb8H6oC1Dp3kjPQJbLEWm5KVJUu6gr9rluUsxlkBG9OXSLFr6fQoOpr0Ihwg/ESbfLbegJXpW3xyoYQkjVOSRkgM+UpMFFCCKsYEFUj2K+YrOqJJGnxUeAcW9N3rGVsUUMYSHh89wsJcy7VA8wv8HM3zBCUk+iQbjLOTF5atJBeNpLmVs7vMr5EoBs3ttYiVAWZgY4IauV5v2D2tCo2l93PaEk1y8x3rNlpggHpV5csZrLzP/9Zk1ZmHm3RNq0xsUBR22b4hj22nR+Qisla4O695PhlstHSZtIbTI3bp836MA713/NAISRjeFTWMjpgv0x2hdlM5jWAijG6RW8L7ViQ1mYlzluBBF3LNqkET0vOfrZ9NT14IBnqVzVmDPl/ircg0ONM4kEINuHLgPZcdo+3NpsMYDSDAKvWyqc6kU8k9Q5E9BA7JwLv8b2ZQkdxOnOaDRGrnnR2sL5N6M+DsGarUYzBGxc55sStdSEMxBtBTxtuP1EkCVC/9kJC9fsk9dWXnbGwb1kG/zsQO+1NXt1aq+XZy5P9k/R5eoNqkb1ONKc2jw6fwg/3pA04GAV2M3CTiFLMMmB4kLmKU467I6wGKUzcc7hV/IZkqChzw+cwSj0ilR5aOQEtfJDEUFX09uWgIBMFu9Sa63oTXk4L16QntjXimESmFud5RLj89GagsHvad6fKtOfE51NAoBNEFhC2Ch6T2fIbyXApnabvs4mp82WKmxvVGYOk4IN/orz0mB8Y2IfkXBbVsYU0vM5DVeBqh66GjPSl8k73cWy+31A9IGfCc/OMJmNaIRB8rAdsbS56e9Ev+jCWJdOL4/kKSE6SlcQT4MCBTpAOzpcoO8kpP025iy4Kf9PkFd+/aW8jtOQ78AOW/ywok+u349AOFGK5t5CzkjGf4f1e8jS35huDcANXnFnsSMR6yLZ06UqtxUEr4iFN0oztHN59NbKFroeYC2KrA16vw0ghcnxgTsvqnLZ7HnYEDsKhSLSM0n6jjgDsk+mQqQo5NnRYc3t8Mtb9GCKTTxE1GuLg0nTbh+hYDW0Tzse7rMDJ5XHEYl0x19EStkxlR1LLIzMWqeSpvQKSqzXV8ARENW9/ZTAc9ztqshBkqZ8yHCfaVQDEiB4FTy3TRrpOsIGErnDS5Hy/Es5g5VYexTLkwDXpoCnO/fwTCEZQG5x0XEgM2EUYToceAu3OGO9Q38jOxS1jNpKbcGPDgbfyuPtqiLwbnXyIZcJH6h7PCwTYfI4AztiM4tukzFM3RanVbNrFvLWnzamHaF/0wzkA1e7Y0FiPrMDHH4ZpiP8Q4Jv5i8uXj4Fgcp5sfeMHDYJ+YfwRGxMF4i/U2hFmoCfni2rG2UTcxrbpv3r7caMVhKj4/0/I49Myzv0M2HfUTYaDT9GsXtYskrTD59dvrM28FkF0yW1A6FND4PPhyHXorXFzUeISKhMV4LhPAso4VIM++18tgWad6TbvdHejgTdYt5GUJVYyH9F3xuFsX+kLIGN/F9Uch1hQ5ggjss9jXQw49Oik9HXsHBtWMBT7DPQBdtfvQeGdhYtLY6LHRLR7em8m4ON99ww9m90+DqpPam0+DIoj3k6l31D/WuCfivDfE3dB8zS4OJh9D/TLmobRblSTB07RoOx9RIBiUbcCJLfTl+mgI8mYA8vaIrMY1lt8idbGbmEE1EIni9VG//JaM1KzEoDSg2qdyUVUW3VBVq4gGKtiYf7mIoghRnN2OEfxsPf5oHIhXlJLrEE2M14yll8ns+hznr3fUQRTKKEzKyPKgaBAycgYmBMNwE8+Uf/6fMptVoJLpfETVHOgNFMHW6HOXNGhEcwvNOCUfDh/+f/FikxwwEOHc6TcbTGHFw73y2QZplmUpDQ4HlRoFj9pIQrS7rxu6XyZuf2NMjpQDtRYCtYxnD9lTdYTtbjdSVDP7UwHP7R+xbQT301psrC4REYPGS2v47hGJSez6fr6VVg6JnwwAYXZRSiEFSed2ZfO6BYpyTvvzeyhOUDiIactwrzatvmhyGtJiotOi3AAMfjDqmxQ3LU4uBNEZTOOcbsIhj6Yfu+iWdc4u9IU5oXF5jjip382JOdU1OWBUFxTcuTC+fai4FEUNYHx+R6Y0qn+kzPOnRyGUw+Px6TE8BuLcLL9LIcVQkLyXJzIzUNwpFRdNhBxcHJUE+V/iAMWranoKvQ7UFQv3RE0Zul1w1zCTU5B5bZWvQDHRDR8cIz0GEV59wPeZLVNxJ4a8VAIpDJU1uXVyR9IjNG4RmiJa/QqstVjO5zkTnTNu2iBI4z3EbsYFK/AmCHjc03kepM1QTzjNrObHR5MT3aig0D1atTbNmTQIjV5YfKsuDfL79towL5MapWrRd+rGIT5RuNCQCfrrTqp97L9qkud48U6i/p5As5mXAPJ3PsceenlDVMhhTOZ9TvTR/LClTfP+o+1Nn3/7aqhhWXkKKuyObmBeugUiVCtjS8FgrygGKxX13RR5TIKQsG4XTVb+cA0/HjbOv3kTJx1drNUTuiHdOSO2qAiNp66qIS4mE+TMJIh5UZf3lXfkJZjugzfvJ73yHol4x/GsC9XYxyTfYa+a2Zy6ccvYA2yQXh4VoNOFiPKghy2gAzprjCwZQxwC3NhbvicJFPg+txCgGz491Bl8l30Igs0dd24m+WN0+9CwZrDnL8yu7vXg5UEBumDqkue3XX/l1v1gXkRlaHAbaNsbv2NIMRjh2+DuQPP0n7wlK08K5a2kGXpdl0u6njD6Gtw/n1CJHoLllxTQMdv4y9K15e7UDJPmMj4QyPu2WBojHDb3hFL3ZMjBK+0RuwAsH01qihq8t1QaIjDeGuW4xERvvGJqid51X61wLXTmmfaQvLdLDJsrdk14IqdNVa5jsBlf3p3UkYoj7lNXh1VNw2b6SiwCUm9oKOKI0FYUvdKRk9DjVw2BGy36A1+MN2Pb31XWKbVnH2+izryOZg08xWJzvs8rALbA8SMIHkx8/BUzSpf7iKnTmSQxKgAVyCIwp2lV0R0+txtcAj76/amDUklm1YdD/hKqlsQHy3Tgwx00EvqnbSiX0uDmIbxeJBu4zkjunALIYLCWeeVdv7qz+0NhEcUq/0gFfBiOSTl0h5Xgy96JBrpYboOmFOi4NYq1xH1u8BplbZyskixPLnHVin0Ds1pGuofwiNgt9ZhMugF1J+VfjBeEqsuE+EVCA3e4ibvJSJyru8d4c1st7T2ZCwWpI3Og8JTQ0QCEGDLFgQCDhaYTjcNFqwcP0HFCgYwHVyuCSatu4m5+jBFbtJ+2VAKmGwbfZAjjriPXeQu0cFHFitEDd/nfPTaJiYsrqM6N9RhdnKSbCtYQmaE4T+Tdc4L55R2BPXzWHYFcyAb2LaCq+46JT3Z9oO7VjOeXANkLIPnBLkYqZ2D0muUFj4Drm7cvjZbMqIDAqCMKaGdpFMZkgeDCAWS0B9CYvkDdKKiEuIwaOJJffnY1gPxua4JNVwRefOxdOSja4J2lwGJWsAiU7BzaWpbYY3Fh5CLDzWTahHRsx665dXIAXOWUbN35mgxG1mA9Y3XnnXdV/Qso4HevWhh2Id5drTkxYS5yggfoRcnC+vZOnO9+2kNGkUMXXKO6jcHkuY0vQ2Ed2RaT0Zhxj+hAY2EPkWUWElIHi0EPCitdoa2glgT3cWN+fsFurL66GOXNCJnI9lS4cxJqjjCa0IgxFcntKRTXXRXjRjNnU7qRj5D3lYpJiFEcTbksAuOK3oTAJNYePw09VqISSqdCnjuhsc1Gza7UNgwgbmNuNINwTpyLlLkraH/DkKU/mL8VhmuvLNja6uVurdpXUPYSd7vYeTs2ZNmNpNooS9tbUbv47veKg94jFjoYD7VPqpjFdLNEcMcIHJ2C5oTUTBv7IgAiN3JvaEKjPfdC7UcmE3VZDpuOfn7lje+knXdgB/PVjjryeXy+hRzcZBNxsPG2DGukeVb3FyTYR/wWdyyOVXw+9vvjDm1cwYCAInFmNY/af+N8QWEk8q1iirTxbCrwHbyfuqHbJFzH516qwLUOyJMiKeVz9FMUS7gPHqSlHzjesQFzqWA2+Fx9iTHUtkK4AYtttfQ8OGUUQcl4z6SE5RkQn/LAzF95k1Ns8MgD7uf21NRQSXaP45sG+hNLXL6FlHhFKjXTUAqVVRVYXRLtkyWoLqLe3BeHj7CGZVS7imkrG/ZQ3GXhuzwsM+saiXPzJHIvjpwMRRUF0eCmAyLjHg4KQbUoqWs8Ooa4bAjHHEX+dwmPVdSl3MNGKvBzdzR4ETc6geaiVBE7iw2DoYu6jao6dnBXbT0UOXbM8eNkynfuJSi7W12xoFxb30hkglEjXdvQWh48f0JPusC4tXyKz04VeXG5rgoDsuFc8k3dD8OgMK6p864BzfmcWpPr4JIw1psGkgScyofQiCJh8GDK/zIJMX8oslY82OCuepHjEQGGfnrIiroPXNIeDntZe05avU5ucXXgPh/V1RHO5ejM60/rCNscnXHxo78aoWVZzZ51qhfXsw6KwXqHZTY+AvbCHDcKIMWzWiTijcFGfXOAgGN4ikbo0BtQ29kbaWDU5eoyjZcdVpV/EiYqqxtf/lUYUOgz0BVtxcMJvujg93vyHi/8CzE+zplIpaKWBzMoKq7TrPH5Syuym2wNkxJX14ZhUUecBzp0EvGv4ETpsqhBAfA5Qieixb+mk7SzLJO3/O57o38oR1g2q2hb+WjZzQ+ejyRf0Z/WiUFvt74R4nUYqaF/RKwXRyL/Jk/0srlaYvbtPb41fkp/6qdGX3g64n8lSP69Be4OM3lpTk3bqBJR3R8dHDhmIfQ5mXFELZi0MGwgBPpP/KJ/80xpHxe4UF9UX4Ry7oED7dY57UoM4ZcCUGDEXigG8vmqoVMFBuf6I74s+t91InqgEHxZEYC3PpuZuUEAvRt8AKnYlThY0XGPhls38kUD6JUBO1hZbzwfLFZRqsFSw7ocLDfCGAMd00c7BisbGf6hmZB/1mOgTBtUVuFHI0Vnc94cWO+3nIu+hyoUKeDMpHnuinVJ1zW+sUTRReC1gk0M3ZBeJhNRIqZlLbGORnjuPq+no6qe0R+dOsNNQ6vE++vz797hEUOOAS9pSSoPTJYCVNzmxWwZpfdsUdalllbZquJWchuM3KVGS2yk3CDati0zoviiN13jGx0OgF++gxif5C4YggWvGX/VAWICIItwdL8Zs4a5y3S5AxKnvWVv6CiqfKGjzvfsENkA9+kQqgNvyhWU0g48L0FmIXil+E2mSRRr4ZajqK5q3E1v8ttI8onl/yhgl1lGdgLG7OmTJyl01sDupFvUTXQYJrVNGXKMNtfE8bqj5dB8KxhP9TiVptZb4JhMwpjjWCQFcL/rkPLOSjw5Mp3oBL/RVz3dZEMYJcE9jYhhHHcYFqpvW82JDDegIhfFMUWS4hinIY5FIIzPyd7/A/ks3Xl1cAAA"""

pathlib.Path("train_baseline.py").write_bytes(
    gzip.decompress(base64.b64decode(TRAINER_B64))
)

if "train_baseline" in sys.modules:
    importlib.reload(sys.modules["train_baseline"])
else:
    import train_baseline

from train_baseline import TrainConfig, TrainResult, train, build_model
from dataclasses import asdict

# FMNIST baixa via torchvision (PyTorch mirrors), não GitHub
print("Trainer OK —", pathlib.Path("train_baseline.py").stat().st_size, "bytes")


In [ ]:
# @title 3. Verificar escala ~107M params
cfg = TrainConfig(width_mult=16.0, hidden_dim=2048)
n = sum(p.numel() for p in build_model(cfg).parameters())
params_m = n / 1e6
print(f"Params: {params_m:.2f}M (alvo ~107M)")
assert 100 <= params_m <= 115, f"Escala fora da faixa: {params_m:.1f}M"


In [ ]:
# @title 4. Rodar 3 configs @107M (GPU)
import json, time
from pathlib import Path
from datetime import datetime, timezone

BASE = dict(
    width_mult=16.0, hidden_dim=2048, batch_size=128,
    lr=0.001, weight_decay=1e-4, device="cuda", amp=True,
    num_workers=2, steps=1000, eval_every=250, log_every=200,
)
WC = dict(warmup_steps=100, scheduler="cosine")
LS = dict(label_smoothing=0.1)
RESULT_DIR = Path("results/100m_colab")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTS = [
    ("exp-100m-wc", "warmup_cosine_100m", "RECOMB exp-007/011", "warmup+cosine (stack base)", {**WC}),
    ("exp-100m-ms", "multi_scale_ls_wc_100m", "NOVEL exp-042/047", "multi_scale + LS + wc", {"mixing": "multi_scale_blend", **LS, **WC}),
    ("exp-100m-tri", "tri_scale_ls_wc_100m", "NOVEL exp-053", "tri_scale + LS + wc", {"mixing": "tri_scale_blend", **LS, **WC}),
]

CPU_BASELINE = 0.8612
CPU_LEADER = 0.8993
LOG_ENTRIES = []
T0_TOTAL = time.time()

def run_one_seed(eid, extra, seed):
    cfg = TrainConfig(seed=seed, **BASE, **extra)
    result = train(cfg)
    payload = asdict(result)
    out = RESULT_DIR / f"{eid}_seed{seed}.json"
    out.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    return payload

for eid, name, nov, lesson, extra in EXPERIMENTS:
    print(f"\n{'='*60}\n{eid} | {lesson}\n{'='*60}")
    t0 = time.time()
    per_seed = []
    for seed in [1000, 1001, 1002]:
        r = run_one_seed(eid, extra, seed)
        per_seed.append(r)
        print(f"  seed {seed}: {r['best_val_acc']*100:.2f}%  ({r['steps_per_sec']:.1f} steps/s)")
    accs = [r["best_val_acc"] for r in per_seed]
    mean = sum(accs) / len(accs)
    std = (sum((a-mean)**2 for a in accs)/max(1,len(accs)-1))**0.5 if len(accs)>1 else 0
    entry = {
        "id": eid,
        "name": name,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "source": "colab_a100_100m",
        "seeds": [1000, 1001, 1002],
        "result": {
            "n_seeds": 3,
            "best_val_acc_mean": mean,
            "best_val_acc_std": std,
            "wall_time_s_mean": sum(r["wall_time_s"] for r in per_seed)/3,
            "steps_per_sec_mean": sum(r["steps_per_sec"] for r in per_seed)/3,
            "per_seed": per_seed,
        },
        "delta_vs_cpu_baseline_pp": round((mean - CPU_BASELINE)*100, 2),
        "delta_vs_cpu_leader_pp": round((mean - CPU_LEADER)*100, 2),
        "novelty_note": nov,
        "lesson": lesson,
    }
    LOG_ENTRIES.append(entry)
    print(f"  >> média {mean*100:.2f}% ± {std*100:.2f}%  Δleader={entry['delta_vs_cpu_leader_pp']:+.2f}pp  ({time.time()-t0:.0f}s)")

out_path = Path("experiments_log_100m_colab.jsonl")
with out_path.open("w") as f:
    for e in LOG_ENTRIES:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")
elapsed = time.time() - T0_TOTAL
print(f"\nSalvo: {out_path.resolve()}")
print(f"Tempo total: {elapsed/60:.1f} min ({elapsed:.0f}s)")


In [ ]:
# @title 5. Ranking + veredito
CPU_WC = 0.8993
CPU_BASE = 0.8612

rows = [(e["id"], e["lesson"], e["result"]["best_val_acc_mean"], e["result"]["best_val_acc_std"], "GPU") for e in LOG_ENTRIES]
rows.sort(key=lambda x: -x[2])

print(f"{'Rank':<5} {'ID':<16} {'Acc':>8} {'±':>6} {'Δ wc':>7} {'Veredito':<14} {''}")
print("-"*72)
for i, (eid, label, acc, std, src) in enumerate(rows, 1):
    d_wc = (acc - CPU_WC) * 100
    if d_wc >= 0.8 and std <= 0.003:
        v = "REVOLUCIONÁRIA"
    elif d_wc >= 0.3:
        v = "PROMISSORA"
    elif d_wc >= -0.29:
        v = "INCONCLUSIVA"
    else:
        v = "REFUTADA"
    print(f"{i:<5} {eid:<16} {acc*100:7.2f}% {std*100:5.2f}% {d_wc:+6.2f}pp {v:<14} {label}")

best = max(LOG_ENTRIES, key=lambda e: e["result"]["best_val_acc_mean"])
b = best["result"]["best_val_acc_mean"]
print(f"\n🏆 Melhor: {best['id']} = {b*100:.2f}%")
if best["id"] in ("exp-100m-ms", "exp-100m-tri") and b > CPU_WC:
    print("→ Stack v2 confirmada: warmup+cosine + multi/tri_scale @107M")
elif best["id"] == "exp-100m-wc":
    print("→ warmup+cosine sozinho continua campeão; mixer não ajudou em 107M")
print(f"\nRef CPU: warmup+cosine {CPU_WC*100:.2f}% | baseline plain {CPU_BASE*100:.2f}%")

In [ ]:
# @title 6. Download resultados
from google.colab import files
files.download("experiments_log_100m_colab.jsonl")